In [1]:
import chess, chess.engine, os, stat
from stockfish import Stockfish
from policy import *
import random

In [7]:
engine_path = r".\stockfish\stockfish\stockfish-windows-x86-64-avx2.exe"
sf = Stockfish(engine_path, parameters={"Threads": 4, "Hash": 256})
sf.set_depth(2)           
sf.set_skill_level(2)
sf.get_engine_parameters()

{'Debug Log File': '',
 'Threads': 4,
 'Hash': 256,
 'Ponder': False,
 'MultiPV': 1,
 'Skill Level': 2,
 'Move Overhead': 10,
 'Slow Mover': 100,
 'UCI_Chess960': False,
 'UCI_LimitStrength': False,
 'UCI_Elo': 1350,
 'Contempt': 0,
 'Min Split Depth': 0,
 'Minimum Thinking Time': 20}

In [4]:
games= load_pgn(r"data\filtered_Romoda.pgn")
agent = Agent("Romoda")
agent.train(games)

Epoch 1/10
137/137 ━━━━━━━━━━━━━━━━━━━━ 178s 1s/step - accuracy: 0.0826 - loss: 6.1487 - val_accuracy: 0.1316 - val_loss: 5.5676
Epoch 2/10
137/137 ━━━━━━━━━━━━━━━━━━━━ 195s 1s/step - accuracy: 0.2515 - loss: 3.7443 - val_accuracy: 0.2044 - val_loss: 5.0663
Epoch 3/10
137/137 ━━━━━━━━━━━━━━━━━━━━ 188s 1s/step - accuracy: 0.5569 - loss: 1.6440 - val_accuracy: 0.2117 - val_loss: 5.4913
Epoch 4/10
137/137 ━━━━━━━━━━━━━━━━━━━━ 165s 1s/step - accuracy: 0.7316 - loss: 0.8930 - val_accuracy: 0.2106 - val_loss: 6.3051
Epoch 5/10
137/137 ━━━━━━━━━━━━━━━━━━━━ 203s 1s/step - accuracy: 0.8300 - loss: 0.5699 - val_accuracy: 0.2184 - val_loss: 6.8417
Epoch 6/10
137/137 ━━━━━━━━━━━━━━━━━━━━ 207s 1s/step - accuracy: 0.8854 - loss: 0.3867 - val_accuracy: 0.2112 - val_loss: 7.4731
Epoch 7/10
137/137 ━━━━━━━━━━━━━━━━━━━━ 182s 1s/step - accuracy: 0.9208 - loss: 0.2784 - val_accuracy: 0.2225 - val_loss: 8.1346
Epoch 8/10
137/137 ━━━━━━━━━━━━━━━━━━━━ 202s 1s/step - accuracy: 0.9430 - loss: 0.2047 - val_accu

In [5]:
board = chess.Board()
def stockfish_move():
    sf.set_fen_position(board.fen())
    move = sf.get_best_move()
    board.push(chess.Move.from_uci(move))
def agent_move():
    move = agent.act(board)
    board.push(move)

In [9]:
NUM_GAMES = 100

with open(r"data\Agent vs Stockfish", "w") as pgn_file:

    for i in range(NUM_GAMES):

        board = chess.Board()
        game = chess.pgn.Game()
        agent_is_white = (random.random()>0.5)

        game.headers["Event"] = "Agent vs Stockfish"
        game.headers["Round"] = str(i + 1)

        if agent_is_white:
            game.headers["White"] = "Agent"
            game.headers["Black"] = "Stockfish"
        else:
            game.headers["White"] = "Stockfish"
            game.headers["Black"] = "Agent"

        node = game

        while not board.is_game_over():

            if board.turn == chess.WHITE:

                if agent_is_white:
                    move = agent.act(board)
                else:
                    sf.set_fen_position(board.fen())
                    best = sf.get_best_move()
                    if best is None:
                        break
                    move = chess.Move.from_uci(best)

            else:  # BLACK to move

                if not agent_is_white:
                    move = agent.act(board)
                else:
                    sf.set_fen_position(board.fen())
                    best = sf.get_best_move()
                    if best is None:
                        break
                    move = chess.Move.from_uci(best)



            board.push(move)
            node = node.add_variation(move)

        game.headers["Result"] = board.result()

        print(game, file=pgn_file)
        print("\n", file=pgn_file)